In [ ]:
from datetime import date
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import os
import pandas as pd

import shutil

***boj-japan***

https://www.boj.or.jp/en/mopo/mpmdeci/state_2025/index.htm

In [ ]:
# boj-japan

# base_url = " "
# source_url = " "

# save_dir = " "
# #pdf_dir = os.path.join(save_dir, "pdfs")

# os.makedirs(save_dir, exist_ok=True)
# #os.makedirs(pdf_dir, exist_ok=True)

session = requests.Session()
# session.headers.update({
#     'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, как Gecko) Chrome/91.0.4472.124 Safari/537.36'
# })

resp = session.get(source_url)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

table = soup.find("table")
rows = table.find_all("tr")[1:]

print(f" {len(rows)} News")

 10 News


In [ ]:

def extract_data(html: str) -> tuple:
    soup2 = BeautifulSoup(html, "html.parser")
    mod_outer_div = soup2.find("div", class_="mod_outer")
    if mod_outer_div:
        paragraphs = mod_outer_div.find_all("p")
        text = "\n".join(p.get_text(strip=True) for p in paragraphs)
    else:
        text = soup2.get_text(separator="\n", strip=True)

    text = text[:500]
    pdf_tag = soup2.find("a", href=re.compile(r"\.pdf$", re.IGNORECASE))
    pdf_path = ""
    if pdf_tag:
        pdf_href = pdf_tag.get("href")
        pdf_url = base_url + pdf_href if pdf_href.startswith("/") else pdf_href
        try:
            pdf_resp = session.get(pdf_url)
            pdf_resp.raise_for_status()
            pdf_filename = os.path.basename(pdf_href)
            pdf_path = os.path.join(SAVE_DIR, pdf_filename)
            with open(pdf_path, "wb") as f:
                f.write(pdf_resp.content)
        except:
            pass


    return text, pdf_path

In [ ]:
results = []

# loop
for row in rows:
    cols = row.find_all("td")
    if len(cols) < 2:
        continue

    date_text = cols[0].get_text(strip=True)
    title = cols[1].get_text(strip=True)
    # <a>
    a_tag = cols[1].find("a")
    if not a_tag:
        continue

    href = a_tag["href"]
    full_url = base_url + href if href.startswith("/") else href

    print(f'{date_text} - {title}')

    if full_url.lower().endswith(".pdf"):
        try:
            pdf_resp = session.get(full_url)
            pdf_resp.raise_for_status()
            pdf_filename = os.path.basename(full_url)
            pdf_path = os.path.join(save_dir, pdf_filename)
            with open(pdf_path, "wb") as f:
                f.write(pdf_resp.content)
            text = "PDF only"
        except Exception as e:
            pdf_path = ""
            text = "PDF unavailable"

        results.append({
            "country": "Japan",
            "source_name": "Bank of Japan",
            "source_url": source_url,
            "doc_url": full_url,
            "doc_type": "PDF",
            "date": date_text,
            "title": title,
            "language": "EN",
            "text": text,
            "file_path": pdf_path
        })

        continue

    # html
    html_resp = session.get(full_url)
    html_resp.raise_for_status()
    html = html_resp.text

    safe_title = title.replace(" ", "_")[:80]
    filename_html = f"{date_text.replace(' ', '_').replace('.', '')}_{safe_title}.html"
    filepath_html = os.path.join(save_dir, filename_html)
    with open(filepath_html, "w", encoding="utf-8") as f:
        f.write(html)

    main_text, pdf_path = extract_data(html)

    if "Statement" in title:
        doc_type = "Statement"
    elif "Plan" in title:
        doc_type = "Guideline"
    else:
        doc_type = "Other"

    results.append({
        "country": "Japan",
        "source_name": "Bank of Japan",
        "source_url": source_url,
        "doc_url": full_url,
        "doc_type": doc_type,
        "date": date_text,
        "title": title,
        "language": "EN",
        "text": main_text,
        "file_path": pdf_path
    })

Oct. 30, 2025 - Statement on Monetary Policy [PDF 169KB]
Sept. 19, 2025 - (Reference) Decisions on Disposal of ETFs and J-REITs (September 2025 MPM) [PDF 97KB]
Sept. 19, 2025 - Statement on Monetary Policy [PDF 227KB]
July 31, 2025 - Statement on Monetary Policy [PDF 255KB]
June 17, 2025 - (Reference) Plan for the Reduction of the Purchase Amount of JGBs (June 2025 MPM) [PDF 438KB]
June 17, 2025 - Statement on Monetary Policy
May   1, 2025 - Statement on Monetary Policy
Mar. 19, 2025 - Statement on Monetary Policy
Jan. 24, 2025 - (Reference) Decision at the January 2025 Monetary Policy Meeting [PDF 435KB]
Jan. 24, 2025 - Change in the Guideline for Money Market Operations


In [ ]:
df_boj = pd.DataFrame(results)
df_boj.to_csv(" .csv", index=False, encoding="utf-8-sig")


***fsa-japan***


https://www.fsa.go.jp/en/news/index.html

In [ ]:
# fsa-japan

# base_url = " "
# source_url = " "

# save_dir = " "
# #pdf_dir = os.path.join(save_dir, "pdfs")

# os.makedirs(save_dir, exist_ok=True)
# #os.makedirs(pdf_dir, exist_ok=True)

session = requests.Session()
# session.headers.update({
#     'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, как Gecko) Chrome/91.0.4472.124 Safari/537.36'
# })

resp = session.get(source_url)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")


items = soup.find_all("li")

print(f" {len(items)} News")


 174 News


In [ ]:

def extract_data(html: str) -> tuple:
    soup2 = BeautifulSoup(html, "html.parser")

    priority_selectors = [
        'p[style*="text-align: justify"]',
        'p.indent.mb0',
        'p.indent'
    ]

    paragraphs = []
    for selector in priority_selectors:
        matches = soup2.select(selector)
        if matches:
            paragraphs = [p.get_text(strip=True) for p in matches if len(p.get_text(strip=True)) > 20]
            if paragraphs:
                break

    if not paragraphs:
        all_p = soup2.find_all("p")
        paragraphs = [p.get_text(strip=True) for p in all_p if len(p.get_text(strip=True)) > 20]

    text = "\n".join(paragraphs)
    text = text[:200] if text else ""
    # text = text[:1000]

    pdf_path = ""
    pdf_tag = soup2.find("a", href=lambda h: h and h.endswith(".pdf"))
    if pdf_tag:
        pdf_url = urljoin(base_url, pdf_tag["href"])
        pdf_filename = os.path.basename(pdf_url)
        pdf_path = os.path.join(pdf_dir, pdf_filename)
        try:
            pdf_resp = session.get(pdf_url, timeout=10)
            pdf_resp.raise_for_status()
            with open(pdf_path, "wb") as f:
                f.write(pdf_resp.content)
        except Exception as e:

            pdf_path = ""

    return text, pdf_path

In [ ]:
results = []
# count = 0

#loop
for item in items:
    # if count >= 33:
    #     break

    # Извлечение даты
    text = item.get_text(strip=True)
    date_match = re.search(r'\(([^)]+2025)\)$', text)
    if not date_match:
        continue

    date_str = date_match.group(1).strip()
    a_tag = item.find("a")
    if not a_tag:
        continue

    # <a>
    title = a_tag.get_text(strip=True)
    href = a_tag.get("href")
    if not href or href.startswith("#"):
        continue

    full_url = urljoin(base_url, href)
    print(f"{date_str} - {title}")

    if full_url.lower().endswith(".pdf"):
        try:
            pdf_resp = session.get(full_url, timeout=10)
            pdf_resp.raise_for_status()
            pdf_filename = os.path.basename(full_url)
            pdf_path = os.path.join(pdf_dir, pdf_filename)
            with open(pdf_path, "wb") as f:
                f.write(pdf_resp.content)
            text = "PDF only"
        except Exception as e:
            print(f" failed： {e}")

        results.append({
            "country": "Japan",
            "source_name": "Financial Services Agency (FSA)",
            "source_url": source_url,
            "doc_url": full_url,
            "doc_type": "PDF",
            "date": date_str,
            "title": title,
            "language": "EN",
            "text": text,
            "file_path": pdf_path
        })

        # count += 1
        continue

    # html
    try:
        html_resp = session.get(full_url, timeout=10)
        html_resp.raise_for_status()
        html = html_resp.text
    except Exception as e:
        continue

    safe_title = re.sub(r'[^\w\-_.]', '_', title)[:80]
    filename_html = f"{date_str.replace(' ', '_').replace(',', '')}_{safe_title}.html"
    filepath_html = os.path.join(save_dir, filename_html)
    with open(filepath_html, "w", encoding="utf-8") as f:
        f.write(html)

    text, pdf_path = extract_data(html)

    # doc type
    if "Policy" in title:
        doc_type = "Policy"
    elif "meeting" in title:
        doc_type = "Report"
    elif "Publication" in title:
        doc_type = "Publication"
    else:
        doc_type = "Others"

    results.append({
        "country": "Japan",
        "source_name": "Financial Services Agency (FSA)",
        "source_url": source_url,
        "doc_url": full_url,
        "doc_type": doc_type,
        "date": date_str,
        "title": title,
        "language": "EN",
        "text": text,
        "file_path": pdf_path
    })

    # count += 1

November 6, 2025 - Publication of Interim Report of the Working Group on Disclosure and Assurance of Sustainability-related Financial Information (of the Financial System Council) and Roadmap for sustainability disclosures and assurance in Japan. (November 6, 2025)
October 28, 2025 - Publication of "FSA Analytical Notes (2025.10): Analysis of Defaults in Housing Loans by Regional Banks" (October 28, 2025)
October 24, 2025 - On the Holding of the Asia GX (Green Transformation) Consortium High-Level Meeting 2025 (October 24, 2025)
October 24, 2025 - Remarks by Minister Katayama during Japan Weeks 2025 (October 24, 2025)
October 23, 2025 - Regarding the Asia Day (October 23, 2025)
October 23, 2025 - The ninth meeting of the Working Group on Disclosure and Assurance of Sustainability-related Financial Information (of the Financial System Council) (October 23, 2025)
October 23, 2025 - Updated : Initiatives by the financial industry to enhance their asset management businesses (October 23, 2